# Screenwise — AI + Health
Use one shared Colab notebook. Run the three code cells in order. No GPU, API key, or private patient data is needed.
Suggested work allocation: Abhi — main implementation (largest); Nehanth185 — evaluation (smallest); Sreel712 — demo and analysis (medium). Each person should review, understand and adapt their part. Follow your competition’s AI-assistance rules.
**This project is a prototype and is not medical advice.**

## Part 1 — Abhinav: Main implementation: data, model and shared functions


In [ ]:
# PART 1 — Abhi: main implementation — data, model and shared functions
# Paste into the FIRST code cell, then run it.
from io import StringIO
from urllib.request import urlopen
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score

DISCLAIMER = 'This project is a prototype and is not medical advice.'
print('SCREENWISE — AI + Health')
print(DISCLAIMER)
print('User: Ana Martinez, a fictional community health nurse in Phoenix.')
print('Moment: reviewing screening follow-ups after a busy clinic session.')
print('Goal: explore which public records a model flags for closer review.')
print('This demonstration must not be used to prioritize actual patients.')

URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
raw = urlopen(URL, timeout=30).read()
columns = ['pregnancies', 'glucose', 'diastolic_bp', 'skin_thickness',
           'insulin', 'bmi', 'pedigree', 'age', 'label']
data = pd.read_csv(StringIO(raw.decode()), header=None, names=columns)
assert data.shape == (768, 9), 'Unexpected dataset shape; check the source.'
assert set(data['label']) == {0, 1}
FEATURES = ['glucose', 'diastolic_bp', 'bmi', 'age']
X = data[FEATURES].copy()
# Zero values in these measurements mean missing data, not healthy values.
X[['glucose', 'diastolic_bp', 'bmi']] = X[['glucose', 'diastolic_bp', 'bmi']].replace(0, np.nan)
y = data['label'].astype(int)
train_ids, test_ids = train_test_split(
    np.arange(len(data)), test_size=0.25, random_state=42, stratify=y)
X_train, X_test = X.iloc[train_ids], X.iloc[test_ids]
y_train, y_test = y.iloc[train_ids], y.iloc[test_ids]
assert set(train_ids).isdisjoint(test_ids)
print(f'Public records: {len(data)} | Train: {len(train_ids)} | Test: {len(test_ids)}')
print('Dataset SHA-256:', hashlib.sha256(raw).hexdigest())
print('Source:', URL)
print('Historical Pima Indians Diabetes dataset, originally attributed to NIDDK.')
print('Population: women aged 21+ of Pima heritage; not representative of everyone.')
print('Target: existing dataset diabetes label, NOT future disease onset.')
display(X_test.head().rename_axis('public_row_index'))


# Train the model and prepare reusable evaluation data.
model = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=1000, random_state=42))
# All preprocessing learns from training data only.
model.fit(X_train, y_train)
scores = model.predict_proba(X_test)[:, 1]
majority_label = int(y_train.mode().iloc[0])
baseline = np.full(len(y_test), majority_label)
THRESHOLD = 0.35  # Illustrative, not clinically chosen or validated.

def metrics(name, predictions):
    tn, fp, fn, tp = confusion_matrix(y_test, predictions, labels=[0, 1]).ravel()
    return {'Method': name, 'Accuracy (%)': round(100*(tp+tn)/len(y_test), 2),
            'True positives': int(tp), 'True negatives': int(tn),
            'Missed cases (FN)': int(fn), 'False alarms (FP)': int(fp),
            'FN rate (%)': round(100*fn/(tp+fn), 2),
            'FP rate (%)': round(100*fp/(tn+fp), 2)}

predictions = (scores >= THRESHOLD).astype(int)
results = pd.DataFrame([metrics('Always-negative baseline', baseline),
                        metrics('Logistic regression', predictions)])





evidence = X_test.copy()
evidence.insert(0, 'record_number', test_ids + 1)
evidence['actual_label'] = y_test.to_numpy()
evidence['model_score'] = scores
evidence['predicted_label'] = predictions
errors = evidence[evidence.actual_label != evidence.predicted_label]


def save_project_outputs(record_number, threshold, input_row, score, flag, actual, comparison, subgroups):
    report = f'''SCREENWISE — AI + HEALTH
    User: Ana Martinez, a fictional community health nurse in a Phoenix teaching-clinic scenario.
    Moment: reviewing screening follow-ups with limited time. This is an educational simulation.
    Inputs: glucose, diastolic blood pressure, BMI, age. Output: model score and review flag.
    Data: public Pima Indians Diabetes dataset only; 768 records. No private patient records.
    Source: {URL}
    Method: stratified 576/192 train/test split, seed 42; training-only median imputation and
    standardization; logistic regression. Zero glucose, BP and BMI values are treated as missing.
    The target is the existing diabetes label, not future onset.
    
    Results at selected threshold {threshold:.2f}:
    {comparison.to_string(index=False)}
    Model ROC AUC: {roc_auc_score(y_test, scores):.4f}; baseline AUC: 0.5000.
    
    Real input example: record {record_number}, {input_row.iloc[0].to_dict()}.
    Score: {score:.6f}, predicted label: {int(flag)}, actual label: {actual}.
    Failure example at the original threshold 0.35: public record 636 has glucose 104,
    BP 72, BMI 31.2, age 38 and score about 0.1774. Its actual label is positive,
    but the model does not flag it. Overlap between classes and omitted clinical factors
    may contribute; the specific cause is unknown.
    Original question: do missed-positive rates differ for ages 21–39 and 40+?
    {pd.DataFrame(subgroups).to_string(index=False)}
    
    Limits: historical data restricted to women aged 21+ of Pima heritage. No validation
    for other populations, no external or prospective evaluation, and no calibration study.
    Threshold 0.35 and the age split are illustrative. Exploring this test set does not
    provide independent threshold validation. Small subgroup differences are uncertain.
    Next: test a locked model and threshold on a compatible, independent public dataset.
    This project is a prototype and is not medical advice.
    '''
    with open('Screenwise_project_writeup.txt', 'w', encoding='utf-8') as f:
        f.write(report)
    export = evidence.copy()
    export['predicted_label'] = (export.model_score >= threshold).astype(int)
    export['threshold'] = threshold
    export.to_csv('Screenwise_public_test_evidence.csv', index=False)

print('Main implementation ready. Run Part 2 next.')


## Part 2 — Nehanth185: Short evaluation and verification


In [ ]:
# PART 2 — Nehanth185: short evaluation and verification
# Paste into the SECOND code cell after running Part 1.
assert 'model' in globals(), 'Run Part 1 first.'
print('MODEL VS SIMPLE BASELINE — threshold 0.35')
display(results.set_index('Method'))
print(f'Model ROC AUC: {roc_auc_score(y_test, scores):.4f}; baseline AUC: 0.5000')
print('Missed cases (FN) are actual positives predicted negative.')
print('False alarms (FP) are actual negatives predicted positive.')
print('FN rate uses actual positives; FP rate uses actual negatives.')
print('Accuracy alone hides the baseline’s 67 missed positives.')

# Check that the test records are separate and every prediction is counted.
assert set(train_ids).isdisjoint(test_ids)
assert len(scores) == len(y_test) == 192
assert np.isfinite(scores).all() and ((scores >= 0) & (scores <= 1)).all()
cm = confusion_matrix(y_test, predictions, labels=[0, 1])
assert int(cm.sum()) == len(y_test)
print('Verification passed. Run Part 3 for the demo and original question.')


## Part 3 — Sreel712: Demo, age analysis and visualization


In [ ]:
# PART 3 — Sreel712: demo, original question and project write-up
# Paste into the THIRD code cell. Run Parts 1 and 2 first.
# Change the two values below and rerun this cell to explore.
record_number = 636 #@param {type:"integer"}
threshold = 0.35 #@param {type:"number"}
assert 'evidence' in globals(), 'Run Parts 1 and 2 first.'
if not 0 <= threshold <= 1:
    raise ValueError('Threshold must be between 0 and 1.')
if record_number not in evidence.record_number.values:
    raise ValueError('Choose a held-out record number from: ' +
                     ', '.join(map(str, evidence.record_number.head(20))))

record = evidence[evidence.record_number == record_number].iloc[0]
# Recompute from actual feature values through the trained pipeline.
input_row = X.loc[[record_number - 1], FEATURES]
score = float(model.predict_proba(input_row)[0, 1])
flag = score >= threshold
actual = int(record.actual_label)
print('SCREENWISE — public-data demonstration')
print(DISCLAIMER)
print(f'Public record: {record_number}')
display(input_row.rename_axis('public_row_index'))
print('Units: glucose = 2-hour OGTT mg/dL; BP = diastolic mm Hg; BMI = kg/m²; age = years.')
print(f'Model score: {score:.1%} | Threshold: {threshold:.2f}')
print('Output:', 'Flag for review' if flag else 'Below flag threshold')
print('Actual dataset label:', 'Positive' if actual else 'Negative')
print('Outcome:', ('True positive' if flag else 'False negative — missed case')
      if actual else ('False positive — false alarm' if flag else 'True negative'))
print('A score is not a validated individual risk estimate or a diagnosis.')
print('A below-threshold score does not rule out diabetes.')

current = (scores >= threshold).astype(int)
comparison = pd.DataFrame([metrics('Always-negative baseline', baseline),
                           metrics('Model at selected threshold', current)])
print('\nTrade-off at your selected threshold:')
display(comparison.set_index('Method'))
print('Changing the threshold reuses the test set: this is exploration, not new validation.')

print('\nOUR QUESTION: Does the model miss more younger or older positive cases?')
subgroups = []
for name, mask in [('Ages 21–39', evidence.age < 40),
                   ('Ages 40+', evidence.age >= 40)]:
    group = evidence.loc[mask]
    positives = group[group.actual_label == 1]
    missed = int((positives.model_score < threshold).sum())
    subgroups.append({'Age group': name, 'Test records': len(group),
                      'Actual positives': len(positives), 'Missed cases': missed,
                      'FN rate (%)': round(100*missed/len(positives), 1)})
display(pd.DataFrame(subgroups).set_index('Age group'))
print('Small exploratory subgroups; differences do not establish fairness or causation.')

# Visualize the trade-off for the selected threshold.
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(['Missed cases (FN)', 'False alarms (FP)'],
       [comparison.iloc[1]['Missed cases (FN)'], comparison.iloc[1]['False alarms (FP)']],
       color=['#d49a64', '#7b9d87'])
ax.set(ylabel='Number of test records', title=f'Errors at threshold {threshold:.2f}')
plt.tight_layout()
plt.show()

save_project_outputs(record_number, threshold, input_row, score, flag,
                     actual, comparison, subgroups)
print('Saved Screenwise_project_writeup.txt and Screenwise_public_test_evidence.csv.')
print('Open the Colab Files sidebar to download your results.')
print('All three parts are complete.')
